# Preparing the Dataset, Scalling, and Walk-forward Spliting to Feed Machines

In [9]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler


In [10]:
# Config

INPUT_FILES = [
    r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process1.csv",
    r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process2.csv",
    r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process3.csv"
]

OUT_DIR = Path(r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)


N_LAGS = 10
SEQ_LEN = 25
HORIZON = 1

WINDOW_SIZE = None # None = Expanding window, or we can set fixed train size, like 200.
STEP = 1           # number of steps to move forward at each winodw
TEST_SIZE = 1      # forecast horizon per rolling step


In [11]:
# helper functions
def make_lag_features(series, n_lags):
    df = pd.DataFrame({"y": series})
    for lag in range(1, n_lags + 1):
        df[f"lag_{lag}"] = df["y"].shift(lag)
    return df.dropna().reset_index(drop=True)

def make_sequence_windows(series, seq_len, horizon):
    x = series.values
    X, Y = [], []
    for i in range(len(x) - seq_len - horizon + 1):
        X.append(x[i:i+seq_len].reshape(seq_len, 1))
        Y.append(x[i+seq_len:i+seq_len+horizon])

    return np.array(X), np.array(Y)

def rolling_windows(n_samples, window_size, step=1, test_size=1):
    """
    Generates start/end indices for rollig/expanding walk-forward windows.
    window_size = None --> expanding window
    """
    starts = []
    ends = []
    idx = window_size if window_size else 0
    while idx + test_size <= n_samples:
        start = 0 if window_size is None else idx - window_size
        end = idx
        starts.append(start)
        ends.append(end)
        idx += step
    return starts, ends


In [12]:
# Main Pipeline
for file in INPUT_FILES:
    print(f"\n=== Processing {file} ===")
    series = pd.read_csv(file).iloc[:, 0] # take first column
    process_name = Path(file).stem
    out_path = OUT_DIR / process_name
    out_path.mkdir(parents=True, exist_ok=True)

    # lag features
    df_lag = make_lag_features(series, N_LAGS)
    n_lag = len(df_lag)
    feature_cols = [c for c in df_lag.columns if c.startswith("lag_")]

    starts, ends = rolling_windows(n_lag, WINDOW_SIZE, STEP, TEST_SIZE)

    for i, (start, end) in enumerate(zip(starts, ends)):
        lag_train = df_lag.iloc[start:end]
        lag_test = df_lag.iloc[end:end+TEST_SIZE]

        if len(lag_train) == 0:
            # skip this window
            continue

        # Scale using only trainig window
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        scaler_X.fit(lag_train[feature_cols])
        scaler_y.fit(lag_train[["y"]])

        # save scalers
        (out_path / "scalers").mkdir(parents=True, exist_ok=True)
        joblib.dump(scaler_X, out_path / "scalers" / f"scaler_X_step{i}.joblib")
        joblib.dump(scaler_y, out_path / "scalers" / f"scaler_y_step{i}.joblib")

        # save scaled date
        df_train_scaled = pd.DataFrame(scaler_X.transform(lag_train[feature_cols]), columns = feature_cols)
        df_train_scaled["y"] = scaler_y.transform(lag_train[["y"]])
        df_train_scaled.to_csv(out_path / f"tabular_train_step{i}.csv", index=False)

        if len(lag_test) >0:
            df_test_scaled = pd.DataFrame(scaler_X.transform(lag_test[feature_cols]), columns=feature_cols)
            df_test_scaled["y"] = scaler_y.transform(lag_test[["y"]])
            df_test_scaled.to_csv(out_path / f"tabular_test_step{i}.csv", index=False)

    print("Lag features with rolling windows OK.")
    
    # sequence windows
    X_seq, Y_seq = make_sequence_windows(series, SEQ_LEN, HORIZON)
    n_seq = len(X_seq)
    starts, ends = rolling_windows(n_seq, WINDOW_SIZE, STEP, TEST_SIZE)

    for i, (start, end) in enumerate(zip(starts, ends)):
        X_train, Y_train = X_seq[start:end], Y_seq[start:end]
        X_test, Y_test = X_seq[end:end+TEST_SIZE], Y_seq[end:end+TEST_SIZE]

        if len(X_train) == 0:
            continue

        # scale using trainig window only
        scaler_seq = StandardScaler()
        scaler_seq.fit(X_train.reshape(-1,1))

        X_train_scaled = scaler_seq.transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
        Y_train_scaled = scaler_seq.transform(Y_train.reshape(-1, 1)).reshape(Y_train.shape)
        np.savez_compressed(out_path / f"seq_train_step{i}.npz", X = X_train_scaled, Y = Y_train_scaled)
        
        if len(X_test) > 0:
            X_test_scaled = scaler_seq.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
            Y_test_scaled = scaler_seq.transform(Y_test.reshape(-1, 1)).reshape(Y_test.shape)
            np.savez_compressed(out_path / f"seq_test_step{i}.npz", X = X_test_scaled, Y = Y_test_scaled)
            
    print("Sequnece windows with rollinng windows OK.")

    # Metadata
    metadata = {
        "file": file,
        "n_lags": N_LAGS,
        "seq_len": SEQ_LEN,
        "horizon": HORIZON,
        "window_size": WINDOW_SIZE,
        "step": STEP,
        "test_size": TEST_SIZE,
        "lag_samples": n_lag,
        "seq_samples": n_seq,
        "n_windows": len(starts)
    }
    joblib.dump(metadata, out_path / "metadata.joblib")
    print(f"Finished {file}.")

print("n=== All processes completed ===")


=== Processing C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process1.csv ===
Lag features with rolling windows OK.
Sequnece windows with rollinng windows OK.
Finished C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process1.csv.

=== Processing C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process2.csv ===
Lag features with rolling windows OK.
Sequnece windows with rollinng windows OK.
Finished C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process2.csv.

=== Processing C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process3.csv ===
Lag features with rolling windows OK.
Sequnece windows with rollinng windows OK.
Finished C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\Raw Data\\process3.csv.
n=== All processes completed ===
